In [ ]:
# ---------------------------------------------------------------------------
# Approach B - local opencode agent server (session-based, NOT OpenAI-compatible)
#
# Run:  opencode serve --port 4096
# Docs: http://localhost:4096/doc
# ---------------------------------------------------------------------------
import os
import httpx
from dotenv import load_dotenv

load_dotenv(override=True)

OPENCODE_SERVER = os.getenv("OPENCODE_SERVER", "http://localhost:4096")

def ask_agent(prompt, provider="opencode", model="deepseek-v4.1-flash-free", base_url=OPENCODE_SERVER):
    with httpx.Client(base_url=base_url, timeout=300) as client:
        session = client.post("/session", json={"title": "python-client"}).json()
        sid = session["id"]
        result = client.post(
            f"/session/{sid}/message",
            json={
                "model": {"providerID": provider, "modelID": model},
                "parts": [{"type": "text", "text": prompt}],
            },
        ).json()
        return "\n".join(p["text"] for p in result["parts"] if p["type"] == "text")

print(ask_agent("what is the date today?"))